In [ ]:
%%bash
#!/bin/bash
#PBS -N iqtree_cons
#PBS -l nodes=1:ppn=8,walltime=1000:00:00
#PBS -j oe
#PBS -o /public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/iqtree/cons_uncons/iqtree.log
#PBS -q low
#PBS -t 1-12%12

cd $PBS_O_WORKDIR

source /public/home/luyt/miniconda3/bin/activate mashtree

# ================== 路径参数 ==================
METADATA="/public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/iqtree/cons_uncons/list.txt"
MAFFT_DIR="/public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/mafft_trim/mafft"
TRIM_DIR="/public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/mafft_trim/trim"
CONSTR_dir="/public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/iqtree/cons_uncons/constraint_tr"  # 约束树文件路径
OUTDIR="/public4/home/luoyt/ralstonia/gene_cluster_tree/statistical_test/do_test/iqtree/cons_uncons/result"
THREADS=8
# ==============================================

# 读取当前任务对应的行
line=$(sed -n "${PBS_ARRAYID}p" $METADATA)
gene=$(echo "$line"  | awk '{print $1}')
seqtype=$(echo "$line" | awk '{print $2}')
model=$(echo "$line"  | awk '{print $3}')

#约束文件
CONSTR="${CONSTR_dir}/${gene}_ralstonia_constraint.tre"
# 根据 seqtype 选输入文件
if [ "$seqtype" == "mafft" ]; then
    inputfile="${MAFFT_DIR}/${gene}_mafft.faa"
else
    inputfile="${TRIM_DIR}/${gene}_trim.faa"
fi

# 输出文件前缀
prefix="${gene}_${seqtype}"
outprefix="${OUTDIR}/${prefix}"

echo "=============================="
echo "Task:    ${PBS_ARRAYID}"
echo "Gene:    ${gene}"
echo "Type:    ${seqtype}"
echo "Model:   ${model}"
echo "Input:   ${inputfile}"
echo "Prefix:  ${outprefix}"
echo "=============================="

# 检查输入文件是否存在
if [ ! -f "$inputfile" ]; then
    echo "[ERROR] 输入文件不存在: $inputfile"
    exit 1
fi

cd ${OUTDIR}

# The best-fit model was selected with IQ-TREE ModelFinder (MFP) and used for both unconstrained and constrained trees.

# 1. 跑无约束树
iqtree -s ${inputfile} -m ${model} \
    --prefix ${outprefix}_unconstr \
    -T ${THREADS} \
    --redo

# 2. 跑有约束树
iqtree -s ${inputfile} -m ${model} \
    -g ${CONSTR} \
    --prefix ${outprefix}_constr \
    -T ${THREADS} \
    --redo

# 3. 合并两棵树
cat ${outprefix}_unconstr.treefile \
    ${outprefix}_constr.treefile \
    > ${outprefix}.trees

# 4. 拓扑检验（AU + RELL + SH）
iqtree -s ${inputfile} -m ${model} \
    -z ${outprefix}.trees \
    -n 0 \
    -zb 1000 \
    -au \
    --prefix ${outprefix}_test \
    -T ${THREADS} \
    --redo

echo "[DONE] ${prefix} 完成"